In [13]:
# ============================================================
# STEP 1: SETUP ENVIRONMENT
# ============================================================
# PURPOSE:
# Import required libraries for data processing and analysis.

import pandas as pd
import numpy as np

# OUTPUT CHECK:
# Libraries imported successfully (no errors expected)

In [14]:
# ============================================================
# STEP 2: LOAD M5 DATASET
# ============================================================
# PURPOSE:
# Load raw M5 dataset which contains demand data in wide format.

m5 = pd.read_csv("../data/sales_train_validation.csv")

# OUTPUT CHECK:
# Expect columns like: id, item_id, store_id, d_1, d_2 ...
print(m5.head())
print("Shape:", m5.shape)

                              id        item_id    dept_id   cat_id store_id  \
0  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
1  HOBBIES_1_002_CA_1_validation  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1   
2  HOBBIES_1_003_CA_1_validation  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1   
3  HOBBIES_1_004_CA_1_validation  HOBBIES_1_004  HOBBIES_1  HOBBIES     CA_1   
4  HOBBIES_1_005_CA_1_validation  HOBBIES_1_005  HOBBIES_1  HOBBIES     CA_1   

  state_id  d_1  d_2  d_3  d_4  ...  d_1904  d_1905  d_1906  d_1907  d_1908  \
0       CA    0    0    0    0  ...       1       3       0       1       1   
1       CA    0    0    0    0  ...       0       0       0       0       0   
2       CA    0    0    0    0  ...       2       1       2       1       1   
3       CA    0    0    0    0  ...       1       0       5       4       1   
4       CA    0    0    0    0  ...       2       1       1       0       1   

   d_1909  d_1910  d_1911  d_1912  d_1913  


In [15]:
# ============================================================
# STEP 3: CONVERT DATA FROM WIDE TO LONG FORMAT
# ============================================================
# PURPOSE:
# Convert daily columns (d_1, d_2...) into rows for time-series analysis.

m5_long = m5.melt(
    id_vars=["id", "item_id", "store_id"],
    var_name="day",
    value_name="demand"
)

# OUTPUT CHECK:
# Expect columns: id, item_id, store_id, day, demand
print(m5_long.head())
print("Shape:", m5_long.shape)

                              id        item_id store_id      day     demand
0  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001     CA_1  dept_id  HOBBIES_1
1  HOBBIES_1_002_CA_1_validation  HOBBIES_1_002     CA_1  dept_id  HOBBIES_1
2  HOBBIES_1_003_CA_1_validation  HOBBIES_1_003     CA_1  dept_id  HOBBIES_1
3  HOBBIES_1_004_CA_1_validation  HOBBIES_1_004     CA_1  dept_id  HOBBIES_1
4  HOBBIES_1_005_CA_1_validation  HOBBIES_1_005     CA_1  dept_id  HOBBIES_1
Shape: (58418840, 5)


In [16]:
# ============================================================
# STEP 4: FIX DATA TYPE FOR DEMAND
# ============================================================
# PURPOSE:
# Ensure demand is numeric for aggregation (mean, std).

m5_long["demand"] = pd.to_numeric(m5_long["demand"], errors="coerce")

# Remove invalid values
m5_long = m5_long.dropna(subset=["demand"])

# OUTPUT CHECK:
print("Demand dtype:", m5_long["demand"].dtype)
print("Missing values:", m5_long["demand"].isna().sum())

Demand dtype: float64
Missing values: 0


In [17]:
# ============================================================
# STEP 5: CREATE DEMAND PROFILES (ITEM-LEVEL STATS)
# ============================================================
# PURPOSE:
# Compute mean and standard deviation per item to classify demand behavior.

item_stats = m5_long.groupby("item_id")["demand"].agg(["mean", "std"]).reset_index()

# OUTPUT CHECK:
print(item_stats.head())
print("Total items:", len(item_stats))

       item_id      mean       std
0  FOODS_1_001  0.640199  1.319567
1  FOODS_1_002  0.383377  0.760323
2  FOODS_1_003  0.700157  1.257818
3  FOODS_1_004  6.768479  8.390284
4  FOODS_1_005  1.197804  2.247593
Total items: 3049


In [18]:
# ============================================================
# STEP 6: CLASSIFY ITEMS INTO FNS (FAST / NORMAL / SLOW)
# ============================================================
# PURPOSE:
# Categorize items based on demand intensity using quantiles.

item_stats["fns_class"] = pd.qcut(
    item_stats["mean"],
    q=3,
    labels=["Slow", "Normal", "Fast"]
)

# OUTPUT CHECK:
print(item_stats["fns_class"].value_counts())

fns_class
Slow      1017
Normal    1016
Fast      1016
Name: count, dtype: int64


In [20]:
# ============================================================
# STEP 7: LOAD SKU MASTER DATASET
# ============================================================
# PURPOSE:
# Load your existing enriched SKU dataset from data folder.

sku = pd.read_csv("../data/sku_master_enriched.csv")

# OUTPUT CHECK:
print(sku.head())
print("Columns:", sku.columns)
print("Shape:", sku.shape)

    sku_id origin_country  unit_cost  annual_demand FNS_class  cum_cost  \
0  SKU_214         Israel      49829             39      Slow     49829   
1   SKU_61         Israel      49740             46      Slow     99569   
2   SKU_52         Russia      49407             44      Slow    148976   
3  SKU_148         France      49095             58      Fast    198071   
4  SKU_101         France      49003             60      Fast    247074   

   cum_cost_pct ABC_class  VED_class  risk_score  base_lead_time_days  \
0      0.006167         A  Essential        0.50                    7   
1      0.012324         A  Essential        0.50                   27   
2      0.018439         A  Essential        0.70                   43   
3      0.024516         A  Essential        0.25                   37   
4      0.030581         A  Essential        0.25                   38   

   distance_km  time_distance_index stocking_policy  baseline_inventory  \
0         1818             0.363673

In [24]:
# ============================================================
# DEBUG: CHECK COLUMN NAMES
# ============================================================
# PURPOSE:
# Identify actual column names in SKU dataset

print(sku.columns)

Index(['sku_id', 'origin_country', 'unit_cost', 'annual_demand', 'FNS_class',
       'cum_cost', 'cum_cost_pct', 'ABC_class', 'VED_class', 'risk_score',
       'base_lead_time_days', 'distance_km', 'time_distance_index',
       'stocking_policy', 'baseline_inventory', 'baseline_inventory_value'],
      dtype='str')


In [25]:
# ============================================================
# STEP: STANDARDIZE COLUMN NAMES
# ============================================================
# PURPOSE:
# Align dataset schema with pipeline expectations

sku = sku.rename(columns={
    "FNS_class": "fns_class",
    "ABC_class": "abc_class",
    "VED_class": "ved_class"
})

# OUTPUT CHECK:
print(sku.columns)
print(sku[["sku_id", "fns_class", "ved_class", "abc_class"]].head())

Index(['sku_id', 'origin_country', 'unit_cost', 'annual_demand', 'fns_class',
       'cum_cost', 'cum_cost_pct', 'abc_class', 'ved_class', 'risk_score',
       'base_lead_time_days', 'distance_km', 'time_distance_index',
       'stocking_policy', 'baseline_inventory', 'baseline_inventory_value'],
      dtype='str')
    sku_id fns_class  ved_class abc_class
0  SKU_214      Slow  Essential         A
1   SKU_61      Slow  Essential         A
2   SKU_52      Slow  Essential         A
3  SKU_148      Fast  Essential         A
4  SKU_101      Fast  Essential         A


In [26]:
# ============================================================
# STEP 8: MAP SKUs TO M5 ITEMS
# ============================================================

mapped_items = []

for fns in ["Slow", "Normal", "Fast"]:
    sku_subset = sku[sku["fns_class"] == fns]
    items_subset = item_stats[item_stats["fns_class"] == fns]["item_id"]
    
    sampled = np.random.choice(items_subset, size=len(sku_subset), replace=True)
    mapped_items.extend(sampled)

sku["mapped_item"] = mapped_items

# OUTPUT CHECK:
print(sku[["sku_id", "fns_class", "mapped_item"]].head())

    sku_id fns_class      mapped_item
0  SKU_214      Slow    HOBBIES_1_422
1   SKU_61      Slow      FOODS_3_427
2   SKU_52      Slow      FOODS_3_597
3  SKU_148      Fast  HOUSEHOLD_2_158
4  SKU_101      Fast  HOUSEHOLD_2_125


In [27]:
print(len(mapped_items), len(sku))

300 300


In [28]:
# ============================================================
# FIX: SAFE FNS-AWARE MAPPING (ROW-ALIGNED)
# ============================================================
# PURPOSE:
# Ensure each SKU gets a correctly aligned mapped_item

sku["mapped_item"] = None

for fns in ["Slow", "Normal", "Fast"]:
    mask = sku["fns_class"] == fns
    items_subset = item_stats[item_stats["fns_class"] == fns]["item_id"]
    
    sampled = np.random.choice(items_subset, size=mask.sum(), replace=True)
    
    sku.loc[mask, "mapped_item"] = sampled

# OUTPUT CHECK:
print(sku[["sku_id", "fns_class", "mapped_item"]].head())
print("Missing mappings:", sku["mapped_item"].isna().sum())

    sku_id fns_class      mapped_item
0  SKU_214      Slow      FOODS_1_151
1   SKU_61      Slow      FOODS_3_779
2   SKU_52      Slow    HOBBIES_1_299
3  SKU_148      Fast  HOUSEHOLD_1_328
4  SKU_101      Fast      FOODS_3_231
Missing mappings: 0


In [31]:
# ============================================================
# STEP 9 (FIXED): MERGE WITH SINGLE STORE
# ============================================================
# PURPOSE:
# Avoid duplication by selecting one store per item

m5_sample = m5_long[
    (m5_long["item_id"].isin(sku["mapped_item"])) &
    (m5_long["store_id"] == "CA_1")   # choose one store
]

merged = sku.merge(
    m5_sample,
    left_on="mapped_item",
    right_on="item_id",
    how="left"
)

# OUTPUT CHECK:
print("Merged shape:", merged.shape)
print("Missing demand:", merged["demand"].isna().sum())
print(merged.head())

Merged shape: (573900, 22)
Missing demand: 0
    sku_id origin_country  unit_cost  annual_demand fns_class  cum_cost  \
0  SKU_214         Israel      49829             39      Slow     49829   
1  SKU_214         Israel      49829             39      Slow     49829   
2  SKU_214         Israel      49829             39      Slow     49829   
3  SKU_214         Israel      49829             39      Slow     49829   
4  SKU_214         Israel      49829             39      Slow     49829   

   cum_cost_pct abc_class  ved_class  risk_score  ...  time_distance_index  \
0      0.006167         A  Essential         0.5  ...             0.363673   
1      0.006167         A  Essential         0.5  ...             0.363673   
2      0.006167         A  Essential         0.5  ...             0.363673   
3      0.006167         A  Essential         0.5  ...             0.363673   
4      0.006167         A  Essential         0.5  ...             0.363673   

   stocking_policy  baseline_invent

In [32]:
# ============================================================
# FINAL VALIDATION BEFORE SAVING
# ============================================================

# 1. Check demand distribution
print(merged["demand"].describe())

# 2. Intermittency check (very important)
print("Zero demand ratio:", (merged["demand"] == 0).mean())

# 3. FNS validation
print("\nFNS vs Demand:")
print(merged.groupby("fns_class")["demand"].mean())

count    573900.000000
mean          1.026919
std           3.163286
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max         138.000000
Name: demand, dtype: float64
Zero demand ratio: 0.651955044432828

FNS vs Demand:
fns_class
Fast      2.509203
Normal    0.614704
Slow      0.229780
Name: demand, dtype: float64


In [30]:
# ============================================================
# STEP 10: VALIDATE DEMAND MAPPING
# ============================================================
# PURPOSE:
# Ensure demand behaves realistically and mapping is correct

# Basic stats
print(merged["demand"].describe())

# Intermittency (important for MRO realism)
print("Zero demand ratio:", (merged["demand"] == 0).mean())

# FNS validation
print("\nAverage demand by FNS class:")
print(merged.groupby("fns_class")["demand"].mean())

count    5.739000e+06
mean     8.815788e-01
std      2.910581e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      3.550000e+02
Name: demand, dtype: float64
Zero demand ratio: 0.697282801881861

Average demand by FNS class:
fns_class
Fast      2.206742
Normal    0.505061
Slow      0.177716
Name: demand, dtype: float64


In [33]:
# ============================================================
# STEP 11: FINAL CLEAN DATASET
# ============================================================

final = merged[[
    "sku_id",
    "fns_class",
    "ved_class",
    "abc_class",
    "day",
    "demand"
]]

# OUTPUT CHECK:
print(final.head())
print("Final shape:", final.shape)

    sku_id fns_class  ved_class abc_class  day  demand
0  SKU_214      Slow  Essential         A  d_1     0.0
1  SKU_214      Slow  Essential         A  d_2     0.0
2  SKU_214      Slow  Essential         A  d_3     0.0
3  SKU_214      Slow  Essential         A  d_4     0.0
4  SKU_214      Slow  Essential         A  d_5     0.0
Final shape: (573900, 6)


In [34]:
# ============================================================
# SAVE FINAL DATASET
# ============================================================

final.to_csv("../data/sku_with_m5_demand.csv", index=False)

print("Day 2 complete")

Day 2 complete


In [35]:
#FOR FUTURE MODELLING:

final["day_num"] = final["day"].str.replace("d_", "").astype(int)